In [1]:
import pandas as pd
import numpy as np
from xbbg import blp
from datetime import datetime, timedelta
from sklearn.linear_model import LinearRegression
from scipy import stats
import matplotlib.pyplot as plt
import re

In [45]:
# -------------------------------------------------------------------------------------------------------------------------------------
# ------------------------------------- SIMPLE SPREAD ANALYSIS - Mean, STD, Corr over 1Y, 5Y, 10Y -------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
def get_vol_spread_table(ccy1: str = 'EURUSD', ccy2: str = 'GBPUSD', 
                          tenors: list = ['1W', '1M', '3M', '6M', '1Y']):
    end_date = datetime.now().strftime("%Y-%m-%d")
    start_date = (datetime.now() - timedelta(days=365 * 11)).strftime("%Y-%m-%d")
    rows = []
    for tenor in tenors:
        ticker1 = f"{ccy1}V{tenor} BGN Curncy"
        ticker2 = f"{ccy2}V{tenor} BGN Curncy"
        try:
            df1 = blp.bdh(ticker1, "PX_LAST", start_date, end_date)
            df2 = blp.bdh(ticker2, "PX_LAST", start_date, end_date)
            df1.columns = [ccy1]
            df2.columns = [ccy2]
            combined = pd.concat([df1, df2], axis=1).dropna()
            spread = combined[ccy1] - combined[ccy2]
            current_spread = spread.iloc[-1]
            # 1Y window (252 trading days)
            c_1y = combined.iloc[-252:] if len(combined) >= 252 else combined
            s_1y = spread.iloc[-252:] if len(spread) >= 252 else spread
            corr_1y = c_1y[ccy1].corr(c_1y[ccy2])
            # 5Y window (1260 trading days)  
            c_5y = combined.iloc[-1260:] if len(combined) >= 1260 else combined
            s_5y = spread.iloc[-1260:] if len(spread) >= 1260 else spread
            corr_5y = c_5y[ccy1].corr(c_5y[ccy2])
            # 10Y window (2520 trading days)
            c_10y = combined.iloc[-2520:] if len(combined) >= 2520 else combined
            s_10y = spread.iloc[-2520:] if len(spread) >= 2520 else spread
            corr_10y = c_10y[ccy1].corr(c_10y[ccy2])
            rows.append({
                'Tenor': tenor,
                '1Y Mean': round(s_1y.mean(), 2),
                '1Y Std': round(s_1y.std(), 2),
                '1Y Corr': round(corr_1y, 2),
                '5Y Mean': round(s_5y.mean(), 2),
                '5Y Std': round(s_5y.std(), 2),
                '5Y Corr': round(corr_5y, 2),
                '10Y Mean': round(s_10y.mean(), 2),
                '10Y Std': round(s_10y.std(), 2),
                '10Y Corr': round(corr_10y, 2),})
        except Exception as e:
            print(f"Error fetching {tenor}: {e}")
    df = pd.DataFrame(rows).set_index('Tenor')
    return df

In [ ]:
tenors = ['1W', '1M', '3M', '6M', '1Y']
df = get_vol_spread_table('AUDUSD', 'NZDUSD', tenors)
df

In [ ]:
# -------------------------------------------------------------------------------------------------------------------------------------
# ---------------------------------- Regression ANALYSIS - CCY1-CCY2 Vol Level relationship -------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------

In [89]:
def parse_lookback(lookback: str) -> tuple:
    period_map = {
        'D': 1,
        'W': 5,
        'M': 21,
        'Y': 252}
    def parse_single_period(period_str: str) -> int:
        match = re.match(r'^(\d+)([DWMY])$', period_str.upper())
        if not match:
            raise ValueError(f"Invalid period format: {period_str}. Use format like '1Y', '3M', '5Y-1Y'")
        num = int(match.group(1))
        unit = match.group(2)
        return num * period_map[unit]
    lookback = lookback.strip().upper()
    if '-' in lookback:
        parts = lookback.split('-')
        if len(parts) != 2:
            raise ValueError(f"Invalid range format: {lookback}. Use format like '5Y-1Y'")
        start_days = parse_single_period(parts[0])  # e.g., 5Y = 1260 days ago
        end_days = parse_single_period(parts[1])    # e.g., 1Y = 252 days ago
        if start_days <= end_days:
            raise ValueError(f"Start period must be further back than end period. Got {parts[0]} to {parts[1]}")
        return (start_days, end_days)
    else:
        start_days = parse_single_period(lookback)
        return (start_days, 0)
def get_lookback_description(lookback: str) -> str:
    start_days, end_days = parse_lookback(lookback)
    if end_days == 0:
        return f"Last {lookback} ({start_days} trading days)"
    else:
        return f"{lookback} ({start_days} to {end_days} trading days ago)"


def fetch_vol_data(ccy_list: list, tenor: str, lookback: str = '10Y'):
    start_days, end_days = parse_lookback(lookback)
    fetch_start = (datetime.now() - timedelta(days=int(start_days * 1.5))).strftime("%Y-%m-%d")
    fetch_end = datetime.now().strftime("%Y-%m-%d")
    data = {}
    for ccy in ccy_list:
        ticker = f"{ccy}V{tenor} BGN Curncy"
        try:
            df = blp.bdh(ticker, "PX_LAST", fetch_start, fetch_end)
            df.columns = [ccy]
            data[ccy] = df[ccy]
        except Exception as e:
            print(f"Error fetching {ccy}: {e}")
    df_full = pd.DataFrame(data).dropna()
    if end_days == 0:
        df_filtered = df_full.iloc[-start_days:]
    else:
        df_filtered = df_full.iloc[-start_days:-end_days]
    return df_filtered

def pairwise_regression(df: pd.DataFrame, ccy1: str, ccy2: str, use_changes: bool = False):
    if use_changes:
        y = df[ccy1].diff().dropna()
        X = df[ccy2].diff().dropna()
        label = "Changes"
    else:
        y = df[ccy1]
        X = df[ccy2]
        label = "Levels"
    aligned = pd.concat([y, X], axis=1).dropna()
    y = aligned.iloc[:, 0].values
    X = aligned.iloc[:, 1].values.reshape(-1, 1)
    model = LinearRegression()
    model.fit(X, y)
    y_pred = model.predict(X)
    residuals = y - y_pred
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (ss_res / ss_tot)
    n = len(y)
    se_beta = np.sqrt(ss_res / (n - 2)) / np.sqrt(np.sum((X - np.mean(X)) ** 2))
    t_stat = model.coef_[0] / se_beta
    p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))
    return {
        'type': label,
        'alpha': model.intercept_,
        'beta': model.coef_[0],
        'beta_se': se_beta,
        't_stat': t_stat,
        'p_value': p_value,
        'r_squared': r_squared,
        'residual_std': np.std(residuals),
        'n_obs': n,
        'start_date': aligned.index[0],
        'end_date': aligned.index[-1]}

def compare_lookback_periods(ccy1: str, ccy2: str, tenor: str, lookbacks: list):
    rows = []
    for lookback in lookbacks:
        try:
            df = fetch_vol_data([ccy1, ccy2], tenor, lookback)
            results = pairwise_regression(df, ccy1, ccy2, use_changes=True)
            rows.append({
                'Lookback': lookback,
                'Start': results['start_date'].strftime('%Y-%m-%d'),
                'End': results['end_date'].strftime('%Y-%m-%d'),
                'Beta': round(results['beta'], 3),
                'R²': round(results['r_squared'], 3),
                'Common %': round(results['r_squared'] * 100, 1),
                'Idio %': round((1 - results['r_squared']) * 100, 1),
                'Resid Std': round(results['residual_std'], 3),
                'N': results['n_obs']
            })
        except Exception as e:
            print(f"Error processing {lookback}: {e}")
    df_summary = pd.DataFrame(rows).set_index('Lookback')
    
    print(f"\n{'='*85}")
    print(f"  Δ{ccy1} ~ β Δ{ccy2} ({tenor}) Pairwise Regression")
    print(f"{'='*85}")
    print(df_summary.to_string())
    print(f"{'='*85}")
    
    return df_summary

In [ ]:
"""
Pairwise Linear Regression Explanation:

Beta: 
    - Sensitivity of Daily Changes in CCY1 Vol on Daily Changes in CCY2 Vol
    -                 "ΔCCY1 ~ β * ΔCCY2"

Common %:     Common Factor Measure
    - Variance of Daily CCY1 Vol Changes explained by CCY2 Vol Changes 
    -            "Changes driven by shared factors"

Resid Std:     Residual Standard Deviation
    - The standard deviation of the regression residuals 
    -        "The typical size of daily idyosyncratic moves"

"""

In [88]:
ccy1 = 'EURUSD'
ccy2 = 'GBPUSD'
tenor = '3M'

df = compare_lookback_periods(ccy1, ccy2, tenor, ['5Y-4Y', '4Y-3Y', '3Y-2Y', '2Y-1Y', '1Y-6M', '6M'])


  ΔEURUSD ~ β ΔGBPUSD (3M) Pairwise Regression
               Start         End   Beta     R²  Common %  Idio %  Resid Std    N
Lookback                                                                        
5Y-4Y     2021-03-16  2022-03-01  0.787  0.556      55.6    44.4      0.107  251
4Y-3Y     2022-03-03  2023-02-16  0.491  0.532      53.2    46.8      0.220  251
3Y-2Y     2023-02-20  2024-02-05  0.868  0.639      63.9    36.1      0.093  251
2Y-1Y     2024-02-07  2025-01-22  0.769  0.515      51.5    48.5      0.125  251
1Y-6M     2025-01-24  2025-07-17  1.202  0.753      75.3    24.7      0.162  125
6M        2025-07-21  2026-01-09  0.775  0.449      44.9    55.1      0.097  125


In [2]:
def EUR_Factors(days):
    tickers = ['EESWE1', 'EESWE2', 'EURUSDV1Y']
    if isinstance(tickers, str):
        tickers = [tickers]
    start_date = (datetime.today() - timedelta(days=days)).strftime('%Y-%m-%d')
    end_date = datetime.today().strftime('%Y-%m-%d')
    out = pd.DataFrame()
    for t in tickers:
        df = blp.bdh(
            tickers=f"{t} BGN Curncy",
            flds=["PX_LAST"],
            start_date=start_date,
            end_date=end_date)
        df = df.droplevel(0, axis=1)
        df = df.rename(columns={"PX_LAST": t})
        df[f"{t}_DayChg"] = df[t].diff()


        df[f"{t}_PctChg"] = df[t].pct_change() 
        out = df if out.empty else out.join(df, how="outer")
    return out


In [10]:
days = 365
df = EUR_Factors(days)

In [11]:
df

,EESWE1,EESWE1_DayChg,EESWE1_PctChg,EESWE2,EESWE2_DayChg,EESWE2_PctChg,EURUSDV1Y,EURUSDV1Y_DayChg,EURUSDV1Y_PctChg
2025-01-13,2.28025,NaN,NaN,2.19600,NaN,NaN,8.1775,NaN,NaN
2025-01-14,2.30300,0.02275,0.009977,2.22775,0.03175,0.014458,8.0450,-0.1325,-0.016203
2025-01-15,2.25025,-0.05275,-0.022905,2.14970,-0.07805,-0.035035,7.9650,-0.0800,-0.009944
2025-01-16,2.22500,-0.02525,-0.011221,2.12725,-0.02245,-0.010443,7.8925,-0.0725,-0.009102
2025-01-17,2.21425,-0.01075,-0.004831,2.11200,-0.01525,-0.007169,7.8725,-0.0200,-0.002534
...,...,...,...,...,...,...,...,...,...
2026-01-06,1.92898,-0.01307,-0.006730,1.99615,-0.02750,-0.013589,6.1100,-0.0350,-0.005696
2026-01-07,1.92034,-0.00864,-0.004479,1.98100,-0.01515,-0.007590,6.1150,0.0050,0.000818
2026-01-08,1.92700,0.00666,0.003468,1.98975,0.00875,0.004417,6.1425,0.0275,0.004497
2026-01-09,1.93810,0.01110,0.005760,2.00715,0.01740,0.008745,6.1675,0.0250,0.004070
